# Manual Component Test Notebook

Test each core module of the Real-Time Speech-to-Speech Translation Service in isolation with your own input files.

- **Goal:** Find bottlenecks and verify correctness for each component.
- **Instructions:**
  - Run each section independently.
  - Change input files/parameters as needed.
  - Observe timing and outputs for each step.

---

## 1. System Audio Capture Handler (Module 6)

Test: Convert an audio file (WebM, WAV, MP3, OGG) to 16kHz mono WAV and run VAD (speech detection).

- Place your test audio file in `input_audio/` or specify a path below.

In [3]:
import sys
import time
from app.system_audio_capture import create_system_audio_handler

# === CONFIG ===
AUDIO_PATH = 'input_audio/speach_sample_1_min.wav'  # Change to your file
SOURCE_FORMAT = 'wav'  # 'webm', 'wav', 'mp3', 'ogg'

handler = create_system_audio_handler()
with open(AUDIO_PATH, 'rb') as f:
    audio_bytes = f.read()

start = time.time()
result = handler.process_system_audio(audio_bytes, SOURCE_FORMAT)
elapsed = (time.time() - start) * 1000

print(f"Success: {result['success']}")
print(f"Has Speech: {result['has_speech']}")
print(f"Energy Level: {result.get('energy_level', None)}")
print(f"Audio Info: {result.get('audio_info', {})}")
print(f"Processing Time: {elapsed:.1f} ms")
if not result['success']:
    print(f"Error: {result.get('error', '')}")

Success: True
Has Speech: False
Energy Level: 0.006514139473438263
Audio Info: {'sample_rate': 16000, 'channels': 1, 'sample_width': 2, 'frames': 955733, 'duration_ms': 59733.3125, 'size_bytes': 1911510}
Processing Time: 27.6 ms


## 2. Streaming Audio Buffer (Module 1)

Test: Segment a WAV file into chunks using the streaming buffer.

- Use the output WAV from the previous step or any 16kHz mono WAV file.

In [11]:
from app.streaming_buffer import StreamingAudioBuffer
import wave
import numpy as np

# === CONFIG ===
WAV_PATH = 'output_audio/test.wav'  # Use result['wav_data'] from above or your own file

# Load WAV file
with wave.open(WAV_PATH, 'rb') as wf:
    sample_rate = wf.getframerate()
    n_frames = wf.getnframes()
    audio = wf.readframes(n_frames)

buffer = StreamingAudioBuffer(max_duration_ms=5000)
start = time.time()
buffer.add_chunk(audio, timestamp=0)
segments = buffer.get_segments()
elapsed = (time.time() - start) * 1000

print(f"Segments: {len(segments)}")
for i, seg in enumerate(segments):
    print(f"  Segment {i}: {len(seg)} bytes")
print(f"Segmentation Time: {elapsed:.1f} ms")

EOFError: 

## 3. Async Processing Queue (Module 2)

Test: Simulate async task queueing and processing. Measures queueing and processing time.

- You can change the number of tasks and worker count below.

In [ ]:
import asyncio
from app.realtime_queue import AsyncProcessingQueue, QueueConfig, ProcessingTask, ProcessingResult
import random

async def dummy_processor(task):
    await asyncio.sleep(random.uniform(0.01, 0.05))  # Simulate work
    return ProcessingResult(task.task_id, success=True, result_data=None)

async def test_queue():
    config = QueueConfig(max_queue_size=10, worker_count=2, task_timeout_sec=1.0)
    queue = AsyncProcessingQueue(config)
    await queue.start_workers(dummy_processor)
    start = time.time()
    for i in range(8):
        task = ProcessingTask(task_id=i, audio_data=b'data', session_id='test', timestamp=time.time())
        await queue.enqueue_task(task)
    await asyncio.sleep(0.2)
    await queue.stop_workers()
    elapsed = (time.time() - start) * 1000
    print(f"Processed 8 tasks in {elapsed:.1f} ms")
    print("Queue stats:", queue.get_queue_stats())

asyncio.run(test_queue())

## 4. Latency Monitor (Module 3)

Test: Wrap any code block to measure stage latency and get stats.

In [ ]:
from app.latency_monitor import LatencyMonitor
import time

monitor = LatencyMonitor(target_latency_ms=2000.0, history_size=100)
monitor.start_session()
with monitor.track_stage("asr"):
    time.sleep(0.05)
with monitor.track_stage("mt"):
    time.sleep(0.03)
with monitor.track_stage("tts"):
    time.sleep(0.04)
session_data = monitor.end_session()
print("Session data:", session_data)
print("Stage stats:", monitor.get_stage_statistics("asr"))
print("Compliance rate:", monitor.get_target_compliance_rate())

## 5. M2M-100 Translation Service (Module 7)

Test: Send a translation request to the running M2M-100 FastAPI service.

- The service must be running (see app/m2m_service.py).
- Change the text, source, and target language as needed.

In [ ]:
import httpx
import time

M2M_URL = 'http://localhost:8001/translate'
session_id = 'manual_test'
text = 'Привет, как дела?'
source_lang = 'ru'
target_lang = 'en'

payload = {
    'session_id': session_id,
    'text': text,
    'source_lang': source_lang,
    'target_lang': target_lang,
    'use_context': True
}
start = time.time()
resp = httpx.post(M2M_URL, json=payload, timeout=10)
elapsed = (time.time() - start) * 1000
if resp.status_code == 200:
    print("Translation:", resp.json().get('translation'))
else:
    print("Error:", resp.text)
print(f"Translation Time: {elapsed:.1f} ms")

## 6. Text-to-Speech (TTS) Module

Test: Convert text to speech using the local TTS engine (pyttsx3).

- Change the text as needed. Output is saved to a WAV file.

In [ ]:
from app.tts import synthesize_text
import time

text = "Hello, this is a test of the TTS module."
output_path = 'output_audio/tts_test.wav'
start = time.time()
audio_bytes = synthesize_text(text)
with open(output_path, 'wb') as f:
    f.write(audio_bytes)
elapsed = (time.time() - start) * 1000
print(f"TTS synthesis complete. Output: {output_path}")
print(f"Synthesis Time: {elapsed:.1f} ms")

## 7. Pipeline Coordinator (Module 4, Optional)

Test: Simulate adding audio chunks and retrieving results from the pipeline coordinator.

- This is an advanced integration test. Requires dummy processor functions.

In [ ]:
from app.pipeline_coordinator import create_pipeline_coordinator, PipelineConfig
import asyncio

async def dummy_processor(task):
    await asyncio.sleep(0.02)
    return {'success': True, 'result_data': b'dummy'}

async def test_coordinator():
    config = PipelineConfig(max_buffer_duration_ms=2000, chunk_size_ms=500, queue_size=5, worker_count=2)
    coordinator = create_pipeline_coordinator('manual_session', config)
    await coordinator.start_pipeline(dummy_processor)
    for i in range(3):
        await coordinator.add_audio_chunk(b'data', 500, True)
    await asyncio.sleep(0.1)
    results = []
    while True:
        res = await coordinator.get_next_result()
        if res is None:
            break
        results.append(res)
    await coordinator.stop_pipeline()
    print(f"Results: {results}")

asyncio.run(test_coordinator())

---
## Notes
- Change file paths and parameters as needed for your tests.
- Each section is independent; you can run them in any order.
- Use the timing output to identify slow/bottleneck components.
- For full pipeline tests, see `scripts/test_realtime_translation.py`.
- For more details, see the technical reports in `docs/technical_reports/`.
